# Gravitational Time Dilation from Wave Equations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Gravitational_Time_Dilation.ipynb)

## What This Notebook Demonstrates

In LFM, lower $\chi$ means lower oscillation frequency. Since $\chi$ is depressed near mass (gravity), waves oscillate **slower** in gravitational wells \u2014 this is **gravitational time dilation**.

We run two identical wave packets:
1. In **flat** $\chi$ (far from mass)
2. In a **$\chi$-well** (near mass)

FFT reveals: $\omega_{\text{well}} < \omega_{\text{flat}}$ \u2014 time runs slower near mass.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
C = 1.0

def laplacian_1d(f, dx):
    return (np.roll(f, 1) + np.roll(f, -1) - 2 * f) / dx**2

# Grid
N = 300
dx = 1.0
dt = 0.02
x = np.arange(N) * dx
n_steps = 4000
center = N // 2

# Chi profiles
chi_flat = np.ones(N) * CHI_0
chi_well = np.ones(N) * CHI_0
well_depth = 3.0
well_width = 30.0
chi_well -= well_depth * np.exp(-(x - center * dx)**2 / (2 * well_width**2))

# Same initial condition for both
E0 = np.exp(-(x - center * dx)**2 / (2 * 10**2))

# Run two simulations in parallel
E_flat = E0.copy()
E_flat_prev = E_flat.copy()
E_well = E0.copy()
E_well_prev = E_well.copy()

hist_flat = np.zeros(n_steps)
hist_well = np.zeros(n_steps)

for step in range(n_steps):
    hist_flat[step] = E_flat[center]
    hist_well[step] = E_well[center]

    E_flat_next = (2*E_flat - E_flat_prev
                   + dt**2 * (C**2 * laplacian_1d(E_flat, dx)
                              - chi_flat**2 * E_flat))
    E_well_next = (2*E_well - E_well_prev
                   + dt**2 * (C**2 * laplacian_1d(E_well, dx)
                              - chi_well**2 * E_well))

    E_flat_prev, E_flat = E_flat, E_flat_next
    E_well_prev, E_well = E_well, E_well_next

# FFT
freqs = np.fft.rfftfreq(n_steps, d=dt)
spec_flat = np.abs(np.fft.rfft(hist_flat))**2
spec_well = np.abs(np.fft.rfft(hist_well))**2

# Find dominant frequency
peak_flat = freqs[np.argmax(spec_flat[1:]) + 1]
peak_well = freqs[np.argmax(spec_well[1:]) + 1]
ratio = peak_well / peak_flat

print("=" * 50)
print("GRAVITATIONAL TIME DILATION")
print("=" * 50)
print(f"  Dominant frequency (flat chi):  {peak_flat:.5f}")
print(f"  Dominant frequency (chi-well):  {peak_well:.5f}")
print(f"  Frequency ratio (well/flat):    {ratio:.5f}")
print(f"\n  Time dilation factor: {ratio:.5f}")
if ratio < 1.0:
    print(f"  CONFIRMED: Waves oscillate SLOWER in chi-well (gravity)")
    print(f"  This IS gravitational time dilation!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: chi profiles
ax = axes[0, 0]
ax.plot(x, chi_flat, 'b-', label='Flat chi', linewidth=2)
ax.plot(x, chi_well, 'r-', label='Chi-well', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('chi(x)')
ax.set_title('Chi Profiles')
ax.legend()
ax.grid(alpha=0.3)

# Top-right: time series
ax = axes[0, 1]
t = np.arange(n_steps) * dt
ax.plot(t[:500], hist_flat[:500], 'b-', alpha=0.7, label='Flat chi')
ax.plot(t[:500], hist_well[:500], 'r-', alpha=0.7, label='Chi-well')
ax.set_xlabel('Time')
ax.set_ylabel('E at center')
ax.set_title('Time Series at Center (first 500 steps)')
ax.legend()
ax.grid(alpha=0.3)

# Bottom-left: FFT spectra
ax = axes[1, 0]
mask = freqs < 2.0
ax.semilogy(freqs[mask], spec_flat[mask], 'b-', alpha=0.7, label='Flat chi')
ax.semilogy(freqs[mask], spec_well[mask], 'r-', alpha=0.7, label='Chi-well')
ax.axvline(peak_flat, color='blue', ls='--', alpha=0.5)
ax.axvline(peak_well, color='red', ls='--', alpha=0.5)
ax.set_xlabel('Frequency')
ax.set_ylabel('Power')
ax.set_title('Frequency Spectra')
ax.legend()
ax.grid(alpha=0.3)

# Bottom-right: summary
ax = axes[1, 1]
ax.bar(['Flat chi', 'Chi-well'], [peak_flat, peak_well],
       color=['blue', 'red'], alpha=0.7)
ax.set_ylabel('Dominant Frequency')
ax.set_title(f'Time Dilation: ratio = {ratio:.4f}')
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Gravitational Time Dilation from GOV-01',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- Waves in a $\chi$-well oscillate at **lower frequency** than in flat $\chi$
- This is exactly **gravitational time dilation**: clocks tick slower near mass
- No metric tensor was injected \u2014 time dilation emerges from the $\chi$-dependent dispersion relation
- The ratio $\omega_{\text{well}}/\omega_{\text{flat}} < 1$ maps directly to the GR time dilation factor